# Hostamar — Free GPU Video Backend (AnimateDiff Lightning)

Run this notebook on Kaggle to expose an HTTP endpoint that the hostamar `gpu-worker` can call.

**Hardware**: 1× Tesla T4 (Kaggle-provided, ~16GB VRAM — fits AnimateDiff Lightning easily).

**What this does**:
1. Installs `diffusers`, `transformers`, `accelerate` on Kaggle
2. Downloads `ByteDance/AnimateDiff-Lightning` from HuggingFace Hub (~3GB checkpoint)
3. Starts a tiny FastAPI/uvicorn HTTP server on a public URL (via ngrok)
4. Exposes `POST /generate` that takes `{prompt, aspect_ratio, duration}` → returns an MP4 file URL

## Setup

Before running:
1. Notebook → Settings (right panel) → Accelerator → **GPU T4×2**
2. Save a version (Quick save)
3. **Cell 4** has a placeholder for your `KAGGLE_VIDEO_TOKEN`. Replace with a random string (e.g. `python -c "import secrets; print(secrets.token_urlsafe(32))"`) — your local worker must use the SAME string. **Do not put your Google account email here.**
4. **Cell 4** also needs an `ngrok` authtoken. Get one free at https://dashboard.ngrok.com/signup → Copy your token. (Optional but required for the public URL — without it kernels can't be reached from outside Kaggle.)

## How long before the notebook is ready

- First install: ~3 min
- AnimateDiff Lightning checkpoint download: ~1-3 min on Kaggle bandwidth
- First test render: ~30s
- **Total cold start: ~5-7 minutes**

## When you're done

Set on your local machine's `.env.local`:

```
KAGGLE_VIDEO_URL=https://<random>.ngrok-free.app/generate
KAGGLE_VIDEO_TOKEN=<the same you put in Cell 4>
```

Then `docker compose -f docker-compose.local.yml up -d --no-deps --build gpu-worker` and enqueue any test job — it'll try Kaggle before Replicate/Fal.

In [ ]:
# Cell 1: confirm GPU
import torch, subprocess
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")
print(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
assert torch.cuda.is_available(), "enable GPU in Notebook settings!!"
# warmup
_ = (torch.zeros(2, 2, device='cuda') + 0).cpu().item()
print("GPU compute OK")


In [ ]:
# Cell 2: install deps — Kaggle has torch pre-installed for GPU,
# we just need diffusers + a small HTTP server.
%pip install --quiet --no-cache-dir \
    diffusers>=0.30.0 \
    transformers>=4.44.0 \
    accelerate>=0.34.0 \
    safetensors>=0.4.5 \
    fastapi>=0.115.0 \
    uvicorn>=0.32.0 \
    pyngrok>=7.2.0 \
    nest-asyncio>=1.6
print("installed")

In [ ]:
# Cell 3: load AnimateDiff Lightning model.
# The '4-step' checkpoint generates a 4-second clip in ~1-3s on T4 (vs ~30s for the
# standard SVD). For video quality vs speed on a 16GB T4, this is the sweet spot.
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

STEP = 4  # Lightning inference steps — 4 is fastest, 8 is higher quality
MODEL_ID = f"ByteDance/AnimateDiff-Lightning/animatediff_lightning_{STEP}_step_diffusion.safetensors"
print(f"downloading motion adapter: {MODEL_ID}")
motion_adapter = MotionAdapter.from_pretrained(MODEL_ID, torch_dtype=torch.float16)

# SD 1.5 backbone (required for AnimateDiff)
print("loading SD 1.5 backbone...")
pipe = AnimateDiffPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    motion_adapter=motion_adapter,
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print("pipeline ready")


In [ ]:
# Cell 4: token + ngrok setup.
# REPLACE the placeholder below with your chosen shared secret.
# Match this exactly in your local .env.local as KAGGLE_VIDEO_TOKEN.
KAGGLE_VIDEO_TOKEN = "REPLACE_ME_WITH_RANDOM_STRING_min32chars"  # <-- change this

# Optional but recommended for a public URL. Get free at dashboard.ngrok.com
# If you don't set this, the notebook only serves traffic inside Kaggle's network — useless to you.
NGROK_TOKEN = "REPLACE_WITH_YOUR_NGROK_TOKEN"

assert KAGGLE_VIDEO_TOKEN != "REPLACE_ME_WITH_RANDOM_STRING_min32chars", "set KAGGLE_VIDEO_TOKEN first"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print(f"token accepted. make sure your local .env.local has the same KAGGLE_VIDEO_TOKEN")

In [ ]:
# Cell 5: HTTP server
import io, os, time, uuid, base64
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="Hostamar Kaggle backend", version="1.0")
OUTPUT_DIR = "/kaggle/working/generated"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PUBLIC_BASE = None  # filled after ngrok tunnel opens

ASPECT_TO_RES = {
    "16:9": (512, 288),
    "9:16": (288, 512),
    "1:1":  (384, 384),
}

@app.get("/health")
def health():
    return {"status": "ok", "model": "AnimateDiff-Lightning", "step": STEP}

@app.post("/generate")
def generate(body: dict, x_worker_token: str | None = Header(default=None)):
    if x_worker_token != KAGGLE_VIDEO_TOKEN:
        raise HTTPException(status_code=403, detail="forbidden")

    prompt = (body.get("prompt") or "").strip()
    if not prompt:
        raise HTTPException(status_code=400, detail="prompt required")
    aspect = body.get("aspect_ratio", "16:9")
    W, H = ASPECT_TO_RES.get(aspect, ASPECT_TO_RES["16:9"])
    duration = max(2, min(6, int(body.get("duration", 4))))

    t0 = time.time()
    out = pipe(
        prompt=prompt,
        num_frames=duration * 8,           # 8 fps * duration seconds
        height=H,
        width=W,
        num_inference_steps=STEP,
        guidance_scale=1.0,                # Lightning works best at CFG=1.0
    )
    frames = out.frames[0]
    t_gen = time.time() - t0

    # Save as MP4
    fname = f"{uuid.uuid4().hex[:8]}_{int(time.time())}.mp4"
    fpath = os.path.join(OUTPUT_DIR, fname)
    # Use imageio if available; else export to gif and convert
    try:
        import imageio
        writer = imageio.get_writer(fpath, fps=8)
        for frame in frames:
            writer.append_data(frame)
        writer.close()
    except Exception:
        # Fallback: save as GIF
        fpath = fpath.replace(".mp4", ".gif")
        frames[0].save(fpath, save_all=True, append_images=frames[1:], optimize=True, duration=125, loop=0)

    print(f"generated {fname} in {t_gen:.1f}s")
    return JSONResponse({
        "videoUrl":     f"{PUBLIC_BASE}/files/{fname}",
        "duration":     duration,
        "provider":     f"kaggle:animatediff-lightning-{STEP}",
        "renderTimeMs": int(t_gen * 1000),
    })

from fastapi.staticfiles import StaticFiles
app.mount("/files", StaticFiles(directory=OUTPUT_DIR), name="files")

print("FastAPI app created, ready to serve")


In [ ]:
# Cell 6: open the public tunnel + start server
import uvicorn, threading, time

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(3)

tunnel = ngrok.connect(8000)
PUBLIC_BASE = tunnel.public_url
print(f"\n🌐 PUBLIC URL: {PUBLIC_BASE}")
print(f"   Set in your .env.local:  KAGGLE_VIDEO_URL={PUBLIC_BASE}/generate")
print(f"\n📝 To stop: interrupt the kernel.")
print(f"   Persistent on Kaggle: keep this notebook running with GPU; use Scheduled
   Notebook Run (free) to restart weekly.")

# Auto-block forever so the server stays up
while True:
    time.sleep(60)


## Smoke test (run on local machine)

Once you have the URL above, run this from your shell:

```bash
curl -X POST $KAGGLE_VIDEO_URL \
  -H "X-Worker-Token: $KAGGLE_VIDEO_TOKEN" \
  -H "Content-Type: application/json" \
  -d '{"prompt":"a dhaka sunset cinematic","aspect_ratio":"16:9","duration":3}'
```

You should get back JSON with `videoUrl` pointing to `/files/<name>.mp4` on the same ngrok URL.

## Troubleshooting

- **Cannot reach from outside?** You forgot to set `NGROK_TOKEN` in Cell 4. Get one at https://dashboard.ngrok.com/get-started/your-authtoken
- **Render takes 60s+?** First call after a `torch.cuda.empty_cache()` is slow. Subsequent calls are ~3s.
- **OOM?** `STEP=2` if 4 fails; or reduce `num_frames` in Cell 5.
- **Hung kernel?** Kaggle kills idle notebooks at ~12h. Use Scheduled Notebook Run → 'weekly' to auto-restart.